# Dual Pipeline Run & Smart Merge

This notebook runs both the **original YOLO pipeline** and the **original Full System pipeline** independently from start to finish on the selected image. Then, it compares their final output dictionaries field-by-field and uses a smart picker to select the best result (e.g. taking the longest valid string if both succeed).


In [47]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import re
import string
import pytesseract
from PIL import Image
from rembg import remove as rembg_remove
from easyocr import easyocr
from datetime import datetime
from ultralytics import YOLO

# Shared OCR Helpers
from ocr_helpers import choose_address, _clean_name, _extract_birthdate_from_id
from ocr_enhancements import validate_national_id
import rotation_app

pytesseract.pytesseract.tesseract_cmd = r'D:\ocr\tesseract\tesseract.exe'
import os
os.environ['TESSDATA_PREFIX'] = r'D:\ocr\tessdata'

# Init models (done once to save time)
YOLO_MODEL_PATH = r"D:\ocr\national_id_ocr\deployment repo\vso-ocr-backend-main\detect_odjects.pt"
print("Loading YOLO model...")
yolo_model = YOLO(YOLO_MODEL_PATH)
print("Loading EasyOCR...")
reader = easyocr.Reader(['ar', 'en'], gpu=False)

_ARABIC_TO_WESTERN = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')


Using CPU. Note: This module is much faster with a GPU.


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loading YOLO model...
Loading EasyOCR...


### Pipeline 1: Original YOLO Pipeline

This is the exact logic from `yolo_front_ocr.py`.


In [48]:
def _extract_digits_yolo(text):
    text = str(text).translate(_ARABIC_TO_WESTERN)
    return re.sub(r'\D', '', text)

def _clean_arabic_yolo(text):
    text = re.sub(r'[^\u0600-\u06FF0-9\s\-]', ' ', text or "")
    return ' '.join(text.split())

def run_yolo_pipeline(file_path, debug=False):
    data = {
        "first name": "0",
        "seconed name": "0",
        "address": "0",
        "id": "0",
        "birthdate": "0",
        "error": 0
    }
    try:
        # Image is already normalized (cropped and rotated) via shared rotation step
        img = cv2.imread(file_path)
        
        results = yolo_model.predict(img, conf=0.25, verbose=False)
        crops = {}
        
        for box in results[0].boxes:
            class_id = int(box.cls[0].item())
            class_name = yolo_model.names[class_id]
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            
            crop = img[y1:y2, x1:x2]
            
            if class_name not in crops or (crop.shape[0]*crop.shape[1] > crops[class_name].shape[0]*crops[class_name].shape[1]):
                crops[class_name] = crop
        
        if 'firstName' in crops:
            fn_text = pytesseract.image_to_string(crops['firstName'], lang='ara', config='--psm 6 --oem 3')
            data['first name'] = _clean_arabic_yolo(fn_text)
            
        if 'lastName' in crops:
            ln_text = pytesseract.image_to_string(crops['lastName'], lang='ara', config='--psm 6 --oem 3')
            data['seconed name'] = _clean_arabic_yolo(ln_text)
            
        if 'address' in crops:
            addr_img = crops['address']
            addr_img = cv2.GaussianBlur(addr_img, (3,3), 0)
            addr_img = cv2.convertScaleAbs(addr_img, alpha=1.3, beta=10)
            addr_text = pytesseract.image_to_string(addr_img, lang='ara', config='--psm 6 --oem 3')
            data['address'] = ' '.join(_clean_arabic_yolo(addr_text).split())
            
        if 'nid' in crops:
            nid_gray = cv2.cvtColor(crops['nid'], cv2.COLOR_BGR2GRAY)
            gauss = cv2.GaussianBlur(nid_gray, (7,7), 0)
            unsharp = cv2.addWeighted(nid_gray, 2, gauss, -1, 0)
            
            nid_results = reader.readtext(unsharp, detail=0, text_threshold=0.25, width_ths=0.8, low_text=0.01)
            
            if len(nid_results) == 1:
                raw_id = nid_results[0]
            elif len(nid_results) == 0:
                raw_id = "0"
            else:
                raw_id = max(nid_results, key=len)
                
            raw_id = _extract_digits_yolo(raw_id)
            
            if len(raw_id) == 14 and raw_id.endswith('2'):
                reversed_id = raw_id[::-1]
                if reversed_id.startswith('2') or reversed_id.startswith('3'):
                    raw_id = reversed_id
                    
            validated_id = validate_national_id(raw_id)
            data['id'] = validated_id
            
            if len(validated_id) == 14 and validated_id[0] in ('2', '3'):
                century = "19" if validated_id[0] == '2' else "20"
                year = century + validated_id[1:3]
                month = validated_id[3:5]
                day = validated_id[5:7]
                data['birthdate'] = f"{year}-{month}-{day}"
    except Exception as e:
        print(f"YOLO pipeline error: {e}")
        data['error'] = 1

    if data['birthdate'] == "0":
        data['error'] = 1

    return data


### Pipeline 2: Original Full System Pipeline

This is the exact logic from `full_system_ocr_with_rotation.ipynb` (wrapped in a clean function).


In [49]:
def run_full_system_pipeline(file_path):
    data = {
        "first name": "0", "seconed name": "0", "address": "0",
        "id": "0", "birthdate": "0", "error": 0
    }
    try:
        # 2. Extract OCR (using the already normalized file_path)
        img = cv2.imread(file_path)
        cropped = img
        w,h,c = cropped.shape
        o = int(w/2); i = int(h/2.5); n = int(h/6)
        cr          = cropped[n-13:i+15, o:]
        cropped_img = cropped[i+8:, o+10:]

        cr_height   = cr.shape[0]; split_point = int(cr_height * 0.52)
        names_region   = cr[0:split_point, :]
        address_region = cr[split_point:, :] 
        address_region = cv2.GaussianBlur(address_region, (3,3), 0)
        address_region = cv2.convertScaleAbs(address_region, alpha=1.3, beta=10)

        text_names   = pytesseract.image_to_string(names_region, lang='ara', config='--psm 11 --oem 3')
        splited_names = text_names.split('\n')

        arabic_digits = ["٠","١","٢","٣","٤","٥","٦","٧","٨","٩"]
        pun = set(string.punctuation)

        s         = easyocr.Reader(['ar','ar'], gpu=False)
        d_names   = s.readtext(names_region, detail=0, text_threshold=0.18, width_ths=0.9, low_text=0.17)
        d_address = s.readtext(address_region, detail=0, text_threshold=0.15, width_ths=0.7, low_text=0.15, paragraph=True)

        state = 0
        if len(text_names.split('\n')) == 4:
            state = 1
            data["first name"]   = splited_names[0] if len(splited_names)>0 else "0"
            data["seconed name"] = splited_names[2] if len(splited_names)>2 else "0"
            address_easyocr      = ' '.join(d_address) if d_address else ""
            data["address"]      = choose_address("", address_easyocr)

            imgs = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2GRAY)
            gauss = cv2.GaussianBlur(imgs, (7,7), 0)
            unsharp = cv2.addWeighted(imgs, 2, gauss, -1, 0)
            o2 = s.readtext(unsharp, detail=0, text_threshold=0.27, width_ths=0.8, low_text=0.008)
            if len(o2)==1:   data["id"] = o2[0]
            elif len(o2)==0: data["id"] = "0"
            else:            data["id"] = max(o2, key=len)
        else:
            state = 2
            imgs  = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2GRAY)
            imgs  = cv2.medianBlur(imgs, 3)
            gauss = cv2.GaussianBlur(imgs, (5,5), 0)
            unsharp = cv2.addWeighted(imgs, 2.2, gauss, -1.2, 0)
            unsharp = cv2.convertScaleAbs(unsharp, alpha=1.4, beta=5)
            d = d_names
            address_easyocr = ' '.join(d_address) if d_address else ""
            for tok in d:
                if tok in arabic_digits: break
                my_data    = ','.join(d)
                split_list = my_data.split(',')
                data["first name"]   = split_list[0] if split_list else "0"
                data["seconed name"] = ','.join(split_list[1:]).replace(","," ").strip() if len(split_list)>1 else "0"
                data["address"]      = choose_address("", address_easyocr).replace("[","").replace("]","").replace("'","")
            o2 = s.readtext(unsharp, detail=0, text_threshold=0.25, width_ths=0.7, low_text=0.01, paragraph=False)
            if o2 is None or d is None: data["error"] = "1"
            elif len(o2)==1:   data["id"] = o2[0]
            elif len(o2)==0:   data["id"] = "0"
            else:              data["id"] = max(o2, key=len)

        if len(str(data["id"])) < 20: data["error"] = "1"
        for c in list(data["first name"]):
            if c in arabic_digits or c in pun:
                data["first name"] = data["first name"].replace(c, "")
        for c in list(data["seconed name"]):
            if c in arabic_digits or c in pun:
                data["seconed name"] = data["seconed name"].replace(c, "")
        if len(data["seconed name"]) <= len(data["first name"]):
            data["first name"], data["seconed name"] = data["seconed name"], data["first name"]

        first_parts  = [p for p in (data["first name"] or "").split() if p]
        second_parts = [p for p in (data["seconed name"] or "").split() if p]
        if len(first_parts) > 3:  data["error"] = "1"
        if len(second_parts) <= 1: data["error"] = "1"

        ar = list('ابتثجحخدذرزسشصضطظعغفقكلمنهوي')
        for c in list(str(data["id"])):
            if c in ar or c in pun:
                data["id"] = str(data["id"]).replace(c, "")

        arabic_string = str(data["id"])
        matches = re.findall(r'[٠-٩]+', arabic_string)
        matches.reverse()
        concatenated = ''.join(matches)
        if concatenated:
            data["id"] = int(concatenated.translate(_ARABIC_TO_WESTERN))

        data["birthdate"] = _extract_birthdate_from_id(data["id"])
        if data["birthdate"] == "0": data["error"] = "1"
        
        data["id"] = validate_national_id(str(data["id"]))
        # Rotation is now tracked centrally, removed from here
        
    except Exception as e:
        print("Full System pipeline error:", e)
        data['error'] = 1

    return data


### Step 3: Smart Output Merger


In [50]:
def _digits_only(s):
    return re.sub(r'\D', '', (s or "").translate(_ARABIC_TO_WESTERN))

def score_name(s):
    if not s or s == "0": return -1
    if any(ch.isdigit() for ch in s): return -1
    
    s_clean = re.sub(r'[\u064B-\u065F]', '', s)
    arabic_chars = len(re.findall(r'[\u0621-\u064A\u067E\u0686\u0698\u06AF]', s_clean))
    spaces = s_clean.count(' ')
    total_len = len(s_clean)
    
    noise = total_len - (arabic_chars + spaces)
    if noise > 0:
        return arabic_chars - (noise * 5)
    return arabic_chars

def score_address(s):
    if not s or s == "0": return -1
    words = s.split()
    if len(words) < 2: return -1
    
    s_clean = re.sub(r'[\u064B-\u065F]', '', s)
    arabic_chars = len(re.findall(r'[\u0621-\u064A\u067E\u0686\u0698\u06AF]', s_clean))
    digits = len(re.findall(r'[0-9\u0660-\u0669]', s_clean))
    spaces = s_clean.count(' ')
    total_len = len(s_clean)
    
    valid_punct = len(re.findall(r'[\-\u0640]', s_clean))
    noise = total_len - (arabic_chars + digits + spaces + valid_punct)
    
    if noise > 0:
        return (arabic_chars + digits) - (noise * 5)
    return arabic_chars + digits

def score_id(s):
    s_str = str(s)
    digits = _digits_only(s_str)
    if len(digits) != 14:
        return -1
    noise = len(s_str) - 14
    if noise > 0:
        return 14 - (noise * 5)
    return 14

import datetime

def bonus_id_date(s):
    s_str = _digits_only(str(s))
    if len(s_str) != 14:
        return 0
    try:
        century = "19" if s_str[0] == '2' else "20"
        year = int(century + s_str[1:3])
        month = int(s_str[3:5])
        day = int(s_str[5:7])
        datetime.date(year, month, day)
        return 5
    except ValueError:
        return 0

def bonus_address_keywords(s):
    if not s or s == "0": return 0
    keywords = ['مركز', 'قسم', 'شارع', 'محافظة', 'مدينة', 'قرية', 'كفر', 'عزبة', 'بلوك', 'مجاورة']
    s_clean = re.sub(r'[ً-ٟ]', '', s)
    bonus = 0
    for kw in keywords:
        if kw in s_clean:
            bonus += 1
    return bonus

GOVERNORATES = {
    "01": "القاهرة", "02": "الاسكندرية", "03": "بورسعيد", "04": "السويس",
    "11": "دمياط", "12": "الدقهلية", "13": "الشرقية", "14": "القليوبية",
    "15": "كفر الشيخ", "16": "الغربية", "17": "المنوفية", "18": "البحيرة",
    "19": "الاسماعيلية", "21": "الجيزة", "22": "بني سويف", "23": "الفيوم",
    "24": "المنيا", "25": "اسيوط", "26": "سوهاج", "27": "قنا",
    "28": "اسوان", "29": "الاقصر", "31": "البحر الاحمر", "32": "الوادي الجديد",
    "33": "مطروح", "34": "شمال سيناء", "35": "جنوب سيناء"
}

def bonus_cross_check(id_str, addr_str):
    if not id_str or not addr_str or id_str == "0" or addr_str == "0": return 0
    id_str = _digits_only(str(id_str))
    if len(id_str) != 14: return 0
    gov_code = id_str[7:9]
    if gov_code in GOVERNORATES:
        gov_name = GOVERNORATES[gov_code]
        addr_clean = re.sub(r'[ً-ٟ]', '', addr_str)
        addr_norm = addr_clean.replace("ة", "ه").replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
        gov_norm = gov_name.replace("ة", "ه").replace("أ", "ا").replace("إ", "ا").replace("آ", "ا")
        if gov_norm in addr_norm:
            return 10
    return 0

def smart_merge(full_data, yolo_data):
    final = {}
    sources = {}

    fid = str(full_data.get('id', '0'))
    yid = str(yolo_data.get('id', '0'))
    fa = str(full_data.get('address', '0'))
    ya = str(yolo_data.get('address', '0'))

    # Calculate bonuses
    f_cross_bonus = bonus_cross_check(fid, fa)
    y_cross_bonus = bonus_cross_check(yid, ya)

    f_id_bonus = bonus_id_date(fid) + f_cross_bonus
    y_id_bonus = bonus_id_date(yid) + y_cross_bonus

    f_addr_bonus = bonus_address_keywords(fa) + f_cross_bonus
    y_addr_bonus = bonus_address_keywords(ya) + y_cross_bonus

    # 1. ID
    fs_score_id = score_id(fid)
    if fs_score_id > 0: fs_score_id += f_id_bonus
    
    ys_score_id = score_id(yid)
    if ys_score_id > 0: ys_score_id += y_id_bonus
    
    if fs_score_id > ys_score_id:
        final['id'] = fid; sources['id'] = "full_system (better score)"
    elif ys_score_id > fs_score_id:
        final['id'] = yid; sources['id'] = "yolo (better score)"
    elif fs_score_id > 0 and ys_score_id > 0:
        final['id'] = fid; sources['id'] = "full_system (tie, both valid)"
    else:
        final['id'] = fid if len(fid) >= len(yid) else yid
        sources['id'] = "none (unvalidated)"

    # 2. Names
    for key in ['first name', 'seconed name']:
        fv = str(full_data.get(key, '0'))
        yv = str(yolo_data.get(key, '0'))
        fs_score = score_name(fv)
        ys_score = score_name(yv)
        
        if fs_score > ys_score:
            final[key] = fv; sources[key] = "full_system (better score)"
        elif ys_score > fs_score:
            final[key] = yv; sources[key] = "yolo (better score)"
        elif fs_score > 0 and ys_score > 0:
            final[key] = fv; sources[key] = "full_system (tie, both valid)"
        else:
            final[key] = fv if len(fv) >= len(yv) else yv
            sources[key] = "none (unvalidated)"

    # 3. Address
    fs_score_addr = score_address(fa)
    if fs_score_addr > 0: fs_score_addr += f_addr_bonus
    
    ys_score_addr = score_address(ya)
    if ys_score_addr > 0: ys_score_addr += y_addr_bonus
    
    if fs_score_addr > ys_score_addr:
        final['address'] = fa; sources['address'] = "full_system (better score)"
    elif ys_score_addr > fs_score_addr:
        final['address'] = ya; sources['address'] = "yolo (better score)"
    elif fs_score_addr > 0 and ys_score_addr > 0:
        final['address'] = fa; sources['address'] = "full_system (tie, both valid)"
    else:
        final['address'] = fa if len(fa) >= len(ya) else ya
        sources['address'] = "none (unvalidated)"

    # Ensure second name is actually longer/different from first name if needed
    if final["seconed name"] != "0" and len(final["seconed name"]) <= len(final["first name"]):
        final["first name"], final["seconed name"] = final["seconed name"], final["first name"]

    # Finalize
    final['birthdate'] = _extract_birthdate_from_id(final['id']) if score_id(final['id']) > 0 else "0"
    final['error'] = 0 if score_id(final['id']) > 0 and final['birthdate'] != "0" else 1
    # Rotation tracked centrally
    final['_sources'] = sources

    return final


### Execution (Dynamic Image Selection)


In [51]:
import result_logger
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

print("Select the FRONT of the ID image...")
path = filedialog.askopenfilename(title="Select FRONT of ID", filetypes=[("Images", "*.jpg *.jpeg *.png *.bmp")])
root.destroy()

def normalize_card(file_path):
    front_img, rotation_applied, _ = rotation_app.auto_rotate_id_from_path(file_path)
    temp_path = "temp_shared_normalized.png"
    cv2.imwrite(temp_path, front_img)
    return temp_path, rotation_applied

if path:
    print(f"\nNormalizing image ONCE (Rotation + Background Removal)...")
    norm_path, rot_applied = normalize_card(path)

    print(f"\nProcessing Full System Pipeline...")
    full_sys_res = run_full_system_pipeline(norm_path)
    full_sys_res['rotation'] = f"{rot_applied}deg"
    
    print(f"Processing YOLO Pipeline...")
    yolo_res = run_yolo_pipeline(norm_path)
    yolo_res['rotation'] = f"{rot_applied}deg"
    
    print(f"\nMerging outputs...")
    final_result = smart_merge(full_sys_res, yolo_res)
    final_result['rotation'] = f"{rot_applied}deg"
    result_logger.log_result("dual_pipeline", path, final_result)
    
    print("\n" + "="*40)
    print("      FULL SYSTEM PIPELINE RESULT")
    print("="*40)
    for k, v in full_sys_res.items():
        print(f"{k:<15}: {v}")
        
    print("\n" + "="*40)
    print("      YOLO PIPELINE RESULT")
    print("="*40)
    for k, v in yolo_res.items():
        print(f"{k:<15}: {v}")

    print("\n" + "="*40)
    print("      FINAL SMART MERGED RESULT")
    print("="*40)
    for k, v in final_result.items():
        if k != "_sources":
            print(f"{k:<15}: {v}")
            
    print("\n" + "="*40)
    print("      SOURCE TRACKING")
    print("="*40)
    for k, v in final_result["_sources"].items():
        print(f"  {k:<13}: {v}")
else:
    print("No image selected.")

Select the FRONT of the ID image...

Normalizing image ONCE (Rotation + Background Removal)...

Processing Full System Pipeline...


Using CPU. Note: This module is much faster with a GPU.


Processing YOLO Pipeline...

Merging outputs...
[Logger] Saved dual_pipeline result to ocr_results.jsonl

      FULL SYSTEM PIPELINE RESULT
first name     : اسرف
seconed name   : امين صلح عبدالمنم
address        : ٤ مهمرعه ملتثى التهمع الاول - المام
id             : 27405251701832
birthdate      : 1974-05-25
error          : 0
rotation       : 0deg

      YOLO PIPELINE RESULT
first name     : شيرف
seconed name   : عبدالمنعم امين صالح
address        : ع ٠ 9 مجمواعة ١١١ مدينتشش التجمع الاول - القاهره
id             : 2201817100574
birthdate      : 0
error          : 1
rotation       : 0deg

      FINAL SMART MERGED RESULT
id             : 27405251701832
first name     : اسرف
seconed name   : عبدالمنعم امين صالح
address        : ع ٠ 9 مجمواعة ١١١ مدينتشش التجمع الاول - القاهره
birthdate      : 1974-05-25
error          : 0
rotation       : 0deg

      SOURCE TRACKING
  id           : full_system (better score)
  first name   : full_system (tie, both valid)
  seconed name : yolo (better sc